In [1]:
import sys
import os
import pandas as pd

src_path = os.path.abspath(os.path.join(os.path.dirname("__file__"), '..', 'src'))
if src_path not in sys.path:
    sys.path.append(src_path)

from dense_index_bge import DenseIndex
from sparse_index import SparseIndex

In [2]:
court_consideration_df = pd.read_csv("../data/court_considerations.csv")
court_consideration_d = dict(zip(court_consideration_df['citation'].tolist(), court_consideration_df['text'].tolist()))

law_df = pd.read_csv("../data/laws_de.csv")
law_d = dict(zip(law_df['citation'].tolist(), law_df['text'].tolist()))

court_doc = [{'citation':citation, 'text':text} for citation,text in zip(court_consideration_df['citation'].tolist(), court_consideration_df['text'].tolist())]
law_doc = [{'citation':citation, 'text':text} for citation,text in zip(law_df['citation'].tolist(), law_df['text'].tolist())]

test_df = pd.read_csv("../data/test_rewrite_001.csv")
_d = {}
for _, row in test_df.iterrows():
    if row['query_id'] not in _d:
        _d[row['query_id']] = [row['query']]
    else:
        _d[row['query_id']].append(row['query'])
test_dict = {k: v for k, v in sorted(_d.items())}

valid_df = pd.read_csv("../data/valid_rewrite_001.csv")

print("data loaded.")

data loaded.


In [3]:
from FlagEmbedding import FlagReranker, BGEM3FlagModel

model = BGEM3FlagModel('/root/.cache/modelscope/hub/models/BAAI/bge-m3', use_fp16=True, show_progress_bar=False)

dense_index = DenseIndex(model, "/root/autodl-fs/bge-processed/_dense_sparse_court/", court_doc)

sparse_index = SparseIndex(model, "/root/autodl-fs/bge-processed/_dense_sparse_court/", court_doc)

reranker = FlagReranker('/root/.cache/modelscope/hub/models/BAAI/bge-reranker-v2-m3', use_fp16=True, normalize=True)

DenseIndex.embeddings:  (2107648, 1024)


In [4]:
test_df = pd.read_csv("../data/test_rewrite_001.csv")
test_id_2_query_d = {}
for query_id, query in zip(test_df['query_id'], test_df['query']):
    test_id_2_query_d[query_id] = query

In [5]:
import metric_utils
import citation_utils
import hits_utils
import reranker_utils
import rrf
from collections import defaultdict
from tqdm.notebook import tqdm

test_multi_question_df = pd.read_csv("../data/test_rewrite_split_question_001.csv")

recall_citations_l = []

query_id_2_query_list = defaultdict(list)

for query_id, query in zip(test_multi_question_df['query_id'], test_multi_question_df['query']):
    query_id_2_query_list[query_id].append(query)

for query_id, query in test_id_2_query_d.items():
    query_id_2_query_list[query_id].append(query) # 完整的问题在最后一个

recall_hits_l = []
for query_id, query_list in query_id_2_query_list.items():
    all_hits = []
    for query in query_list:
        hits1 = dense_index.search_with_score(query, top_k=500)
        hits2 = sparse_index.search_with_score(query, top_k=500)
        hits_merge = hits_utils.merge_hits_with_score_l_by_max(hits1, hits2)
        all_hits = hits_utils.merge_hits_with_score_l_by_max(all_hits, hits_merge)
        
    print(f"[{query_id}] hits_merge.len:", len(all_hits))

    recall_hits_l.append([hit for hit, score in all_hits])
    

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


[test_001] hits_merge.len: 1935
[test_002] hits_merge.len: 1730
[test_003] hits_merge.len: 1774
[test_004] hits_merge.len: 1677
[test_005] hits_merge.len: 1594
[test_006] hits_merge.len: 2259
[test_007] hits_merge.len: 2570
[test_008] hits_merge.len: 1451
[test_009] hits_merge.len: 1752
[test_010] hits_merge.len: 1807
[test_011] hits_merge.len: 1817
[test_012] hits_merge.len: 2607
[test_013] hits_merge.len: 1512
[test_014] hits_merge.len: 1559
[test_015] hits_merge.len: 2023
[test_016] hits_merge.len: 1979
[test_017] hits_merge.len: 1623
[test_018] hits_merge.len: 2579
[test_019] hits_merge.len: 1876
[test_020] hits_merge.len: 2096
[test_021] hits_merge.len: 2228
[test_022] hits_merge.len: 1520
[test_023] hits_merge.len: 1360
[test_024] hits_merge.len: 2225
[test_025] hits_merge.len: 1970
[test_026] hits_merge.len: 2095
[test_027] hits_merge.len: 1438
[test_028] hits_merge.len: 1661
[test_029] hits_merge.len: 2527
[test_030] hits_merge.len: 1658
[test_031] hits_merge.len: 1184
[test_03

In [9]:
query_id_l = []
result_citation_l = []

for idx, (query_id, query_list) in tqdm(enumerate(query_id_2_query_list.items()), total=len(query_id_2_query_list)):
    # 计算evidence分数
    evidence_l = []
    for hit in recall_hits_l[idx]:
        cc = citation_utils.parse_cc_output_citations_and_sentences(hit['text'])
        for citation, first_appears_idx in cc['citations']:
            evidence = citation_utils.build_evidence(cc['sentences'], first_appears_idx, window_size=5)
            evidence_l.append({'citation':citation, 'text':evidence})

    print("===>",query_id, ", evidence_l.len:", len(evidence_l))

    raw_query = query_list[-1]
    normalized_query_list = query_list[:-1] # 最后一个包含多个问题
    question_l = []
    for q in normalized_query_list:
        _, question = q.rsplit('\n\n', 1)
        question_l.append(question)
    
    evidence_with_score_l_raw = reranker_utils.rerank_by_batch_chunked2(reranker, raw_query, evidence_l)
    # evidence_with_score_l_question = reranker_utils.rerank_by_batch_chunked2(reranker, '\n\n'.join(question_l), evidence_l)
    # evidence_with_score_l = [(raw_score[0], raw_score[1]+0.5*question_score[1]) for raw_score, question_score in zip(evidence_with_score_l_raw, evidence_with_score_l_question)]
    
    evidence_with_score_l = evidence_with_score_l_raw
    
    # agg by sum
    citation_score_d = defaultdict(float)
    for evidence, score in evidence_with_score_l:
        citation_score_d[evidence['citation']] += score
        
    # 对综合分数排序
    sorted_citation_score_l = sorted([(c,score) for c,score in citation_score_d.items()], key=lambda x: x[1], reverse=True)
    
    # 将citation合并
    query_result = [citation for citation, _ in sorted_citation_score_l]
    
    query_result2 = [citation for citation in query_result if citation in court_consideration_d or citation in law_d]
    
    print("query_result2:", len(query_result2))
    
    result_citation_l.append(';'.join(query_result2[:100]))
    query_id_l.append(query_id)
    
    print(f"{query_id} query_result2.len:", len(query_result2))


result_df = pd.DataFrame({'query_id':query_id_l, 'predicted_citations':result_citation_l})
result_df.to_csv("../output/submission_100.csv", index=False)

  0%|          | 0/40 [00:00<?, ?it/s]

===> test_001 , evidence_l.len: 4316


KeyboardInterrupt: 

In [14]:
limit=15
sub_df = pd.read_csv("../output/submission_100.csv")
pred_l = []
for query_id, pred in zip(sub_df['query_id'].tolist(), sub_df['predicted_citations'].tolist()):
    pred2 = pred.split(";")[:limit]
    pred_l.append(';'.join(pred2))
new_sub_df = pd.DataFrame({'query_id':sub_df['query_id'].tolist(), 'predicted_citations':pred_l})
new_sub_df.to_csv("../output/submission.csv", index=False)